In [2]:
import pandas as pd
import platform
import os
import sqlite3
import numpy as np
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [3]:
if "pop-os" in platform.node():
    ROOT = r"/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/"
else:
    ROOT = r'/gpfs/home4/athamma/repo/ss-llm/nanoGPT/'
    
TOKENIZER_ROOT = os.path.join(ROOT, "data")
OUT_ROOT = os.path.join(ROOT, "output_dump")
RESULTS_ROOT = os.path.join(ROOT, "results")

SQL_DB = os.path.join(RESULTS_ROOT, "results.db")

def create_connection_cursor(db_file):
    """
    Create a database connection to the SQLite database specified by the db_file

    Args:
        db_file (str): database file

    Returns:
        Connection object or None
    """
    conn = sqlite3.connect(db_file)
    c = conn.cursor()
    return conn, c

conn, c = create_connection_cursor(SQL_DB)

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.mixed_linear_model import MixedLM
import scipy.stats as stats

# Create sample data
np.random.seed(42)
n_subjects = 30
n_observations = 1000

data = pd.DataFrame({
    'subject': np.random.randint(1, n_subjects+1, n_observations),
    'word': np.random.choice(['the', 'dog', 'cat', 'ran', 'jumped'], n_observations),
    'word_length': np.random.randint(2, 8, n_observations),
    'position': np.random.randint(1, 10, n_observations),
    'surprisal': np.random.normal(0, 1, n_observations),
    'reading_time': np.random.normal(300, 50, n_observations)
})

# Create word type IDs
data['word_type_id'] = pd.Categorical(data['word']).codes

# Center and scale predictors
for col in ['word_length', 'position', 'surprisal']:
    data[f'{col}_c'] = (data[col] - data[col].mean()) / data[col].std()

# Create design matrices for random effects
# For subject random effects (slopes and intercepts)
subj_fe = sm.add_constant(data[['word_length_c', 'position_c']])  # for baseline model
subj_fe_full = sm.add_constant(data[['word_length_c', 'position_c', 'surprisal_c']])  # for full model

# Fit baseline model
baseline_model = MixedLM(
    endog=data['reading_time'],
    exog=subj_fe,
    groups=data['subject'],
    exog_re=subj_fe  # This specifies random slopes for all fixed effects
)
baseline_result = baseline_model.fit()

# Fit full model
full_model = MixedLM(
    endog=data['reading_time'],
    exog=subj_fe_full,
    groups=data['subject'],
    exog_re=subj_fe_full  # Random slopes for all fixed effects including surprisal
)
full_result = full_model.fit()

# Calculate and compare log-likelihoods
baseline_ll = baseline_result.llf
full_ll = full_result.llf
delta_ll = full_ll - baseline_ll

# Likelihood ratio test
df = 1  # difference in parameters
lr_stat = 2 * delta_ll
p_value = stats.chi2.sf(lr_stat, df)

# Print results
print("Model Comparison Results:")
print(f"Baseline Log-Likelihood: {baseline_ll:.2f}")
print(f"Full Model Log-Likelihood: {full_ll:.2f}")
print(f"Improvement (ΔLL): {delta_ll:.2f}")
print(f"Likelihood Ratio Test Statistic: {lr_stat:.2f}")
print(f"p-value: {p_value:.4f}")

# Print model summaries
print("\nBaseline Model Summary:")
print(baseline_result.summary())
print("\nFull Model Summary:")
print(full_result.summary())

/home/abishekthamma/PycharmProjects/masters_thesis/mt1/lib/python3.9/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/abishekthamma/PycharmProjects/masters_thesis/mt1/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
/home/abishekthamma/PycharmProjects/masters_thesis/mt1/lib/python3.9/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/abishekthamma/PycharmProjects/masters_thesis/mt1/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with cg
  warnings.warn(
/home/abishekthamma/PycharmProjects/masters_thesi

Model Comparison Results:
Baseline Log-Likelihood: -5311.48
Full Model Log-Likelihood: -5309.58
Improvement (ΔLL): 1.91
Likelihood Ratio Test Statistic: 3.82
p-value: 0.0507

Baseline Model Summary:
                    Mixed Linear Model Regression Results
Model:                   MixedLM       Dependent Variable:       reading_time
No. Observations:        1000          Method:                   REML        
No. Groups:              30            Scale:                    2399.4262   
Min. group size:         20            Log-Likelihood:           -5311.4837  
Max. group size:         50            Converged:                No          
Mean group size:         33.3                                                
-----------------------------------------------------------------------------
                                Coef.  Std.Err.    z    P>|z|  [0.025  0.975]
-----------------------------------------------------------------------------
const                          300.328   

In [5]:
#Retreive the data from the database for subject reading times, join with word details

baseline_df = pd.read_sql_query("""
SELECT SPRTNaturalStories.RTUID, SPRTNaturalStories.WorkerID, SPRTNaturalStories.StoryWordID, SPRTNaturalStories.RT, WordDetails.Word as WordCategory, WordDetails.CharacterLength, WordDetails.WordUID as WordCategoryID
FROM SPRTNaturalStories
JOIN Story on SPRTNaturalStories.StoryWordID = Story.StoryWordID
JOIN WordDetails on WordDetails.WordUID = Story.WordUID
ORDER BY SPRTNaturalStories.WorkerID """ , conn)

#Log transform the reading times
baseline_df['LogRT'] = np.log(baseline_df['RT'])
baseline_df["WorkerCategory"] = pd.Categorical(baseline_df['WorkerID']).codes
baseline_df["WordCategoryID"] = baseline_df["WordCategoryID"].astype('int')

#Center and scale the word length
baseline_df['CharacterLength_c'] = (baseline_df['CharacterLength'] - baseline_df['CharacterLength'].mean()) / baseline_df['CharacterLength'].std()



baseline_df.head()

,RTUID,WorkerID,StoryWordID,RT,WordCategory,CharacterLength,WordCategoryID,LogRT,WorkerCategory,CharacterLength_c
0,66,A117RW2F1MNBQ8,1,360.0,if,2,1,5.886104,0,-1.062253
1,152,A117RW2F1MNBQ8,2,304.0,you,3,2,5.717028,0,-0.620433
2,240,A117RW2F1MNBQ8,3,270.0,were,4,3,5.598422,0,-0.178614
3,326,A117RW2F1MNBQ8,4,292.0,to,2,4,5.676754,0,-1.062253
4,415,A117RW2F1MNBQ8,5,304.0,journey,7,5,5.717028,0,1.146844


In [6]:
for col in baseline_df.columns:
    print(col, baseline_df[col].dtype)

RTUID int64
WorkerID object
StoryWordID int64
RT float64
WordCategory object
CharacterLength int64
WordCategoryID int64
LogRT float64
WorkerCategory int16
CharacterLength_c float64


In [7]:

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# data = sm.datasets.get_rdataset("dietox", "geepack").data
# md = smf.mixedlm("Weight ~ Time", data, groups=data["Pig"])
# mdf = md.fit(method=["lbfgs"])
# print(mdf.summary())


In [8]:
#only random intercept for each subject and word-type (word-type being POS tag I guess)  from baseline_df 

#Randomly Select 1/4th of data

fit_df = baseline_df.sample(frac=0.005)

# md = sm.MixedLM.from_formula("LogRT ~ 1", groups = np.ones(fit_df.shape[0]), vc_formula={"0+C(WorkerID)+C(WordCategoryID)": "0+C(WorkerID)+C(WordCategoryID)"}, data=fit_df)
# mdf = md.fit()

# print(mdf.summary())



In [9]:
md = sm.MixedLM.from_formula("LogRT ~ 1", groups = np.ones(fit_df.shape[0]), 
                             vc_formula={"w1": "0 + C(WorkerID)",
                                         "w2": "0 + C(WordCategoryID)"
                             }, data = fit_df
)
# mdf = md.fit()

# print(mdf.summary())

In [10]:

#import pymer4
from pymer4.models import Lmer


In [12]:
fit_formula = "LogRT ~ 1  + ( 1 | WorkerID) + (1 | WordCategoryID)"

# Fit the model
lm = Lmer(fit_formula, data=fit_df)
lm.fit()


: 

NameError: name 'fit_df' is not defined